# 01 - Data Understanding

**Objectif :** charger le dataset initial, créer immédiatement un train/test brut, puis inspecter uniquement le train.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Data source

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

loader = CsvLoanDataLoader(
    path=settings.raw_data_path,
    sep=settings.raw_data_sep,
    encoding=settings.raw_data_encoding,
)

raw_df = loader.load()
raw_df.head()

## 2. Initial raw train/test split

In [ ]:
from credit_risk_lab.application import DatasetSplitter, SplitConfig
from credit_risk_lab.infrastructure.data_sources import CSVDatasetRepository

initial_split_config = SplitConfig(
    test_size=0.10,
    random_state=settings.random_state,
    stratify=True,
)
initial_splitter = DatasetSplitter(initial_split_config)

raw_train_df, raw_test_df = initial_splitter.split(raw_df)
split_summary = initial_splitter.summary(raw_train_df, raw_test_df)

CSVDatasetRepository.save(raw_train_df, settings.raw_train_path)
CSVDatasetRepository.save(raw_test_df, settings.raw_test_path)

split_summary

## 3. Initial leakage check

In [ ]:
from credit_risk_lab.infrastructure.analytics import DataLeakageAuditor

leakage_auditor = DataLeakageAuditor(target_column=settings.target_column)
leakage_auditor.row_overlap_report(
    raw_train_df,
    raw_test_df,
    holdout_name="raw_test",
)

## 4. Train-only dataset inspection

In [ ]:
from credit_risk_lab.infrastructure.analytics import DatasetInspector

inspector = DatasetInspector(raw_train_df)
summary = inspector.summary()
summary

## 5. Train-only column summary

In [ ]:
column_summary = inspector.column_summary(sample_size=2)
column_summary

## 6. Train-only target distribution

In [ ]:
target_distribution = inspector.target_distribution(settings.target_column)
display(target_distribution)

from credit_risk_lab.infrastructure.visualization import plot_target_distribution

plot_target_distribution(target_distribution).show()